# Setup

In [ ]:
# Auto-install anything missing, as 02b does. gt/gtExtras/gtsummary are the usual
# culprits on a fresh cloud environment. Note lme4 Depends on Matrix, so attaching it
# here also makes S4 subsetting of the kinship matrix work.
pkgs <- c('data.table','ggplot2','ggrepel','gt','gtExtras','gtsummary',
          'lme4','lme4qtl','lmerTest','lmtest','parallel')
if(!require(lme4qtl)) { remotes::install_github("variani/lme4qtl") } # Special install
lapply(pkgs, \(pkg) { if(!require(pkg, character.only=T)) {install.packages(pkg);require(pkg,character.only=T)}}) |> invisible()

options(datatable.na.strings=c('NA',''))

# Every input this notebook does not produce itself is pulled from the workspace
# bucket, so a fresh cloud environment can run it without depending on files an
# earlier session happened to leave on the persistent disk.
ws_bucket <- Sys.getenv('WORKSPACE_BUCKET',
                        unset='gs://fc-secure-4a392455-5587-4d6f-b8bd-01a1f834ae63')
grab_file_if_not_extant <- \(f) if(!file.exists(f)) {
  dir.create(dirname(f), recursive=TRUE, showWarnings=FALSE)
  system(paste0('gcloud storage cp ', ws_bucket, '/', f, ' ', f)) # Needs a local destination, not just the source.
}

grab_file_if_not_extant('data/derived/analysis_df-fhs.csv')                       # 03b
grab_file_if_not_extant('data/derived/genomics/fhs_km.RData')                     # 03b
grab_file_if_not_extant('data/derived/metabolomics/QCd/merged_QCd-aligned.csv')   # 03a
grab_file_if_not_extant('data/derived/metabolomics/met_info-aligned.csv')         # 03a
grab_file_if_not_extant('data/derived/metabolomics/met_info-mesa.csv')            # 01a

# Workaround so that lmerTest works on lme4qtl model objects, so we can get lmerTest Satterthwaite p-values.
myLmer <- \(formula, data, relmat) {
  # method.relfac='chol' skips relfac's duplicated-column scan, which extracts every
  # column of the kinship matrix as a dense vector on EVERY fit -- far more expensive
  # than the factorization itself, and it can divert to a dense eigendecomposition.
  # Results are bit-identical.
  model <- relmatLmer(formula, data, relmat=relmat, method.relfac='chol')
  lmerTest:::as_lmerModLT(model, as.function(model))
}

In [ ]:
data <- fread('data/derived/analysis_df-fhs.csv')
variants_of_interest <- fread(cmd='gcloud storage cat gs://fc-secure-4a392455-5587-4d6f-b8bd-01a1f834ae63/variants_of_interest.csv')
cpaids <- variants_of_interest[cpaid %in% names(data), cpaid]

data <- fread('data/derived/analysis_df-fhs.csv')[
  ][, metabolomics_visit := as.factor(metabolomics_visit)
  # Variants are numeric (0/1/2), but we temporarily set to character so they don't get scale()'d.
  ][, names(.SD) := lapply(.SD, as.character), .SDcols=cpaids
  ][, names(.SD) := lapply(.SD, scale       ), .SDcols=is.numeric
  ][, names(.SD) := lapply(.SD, as.numeric  ), .SDcols=cpaids
]

load('data/derived/genomics/fhs_km.RData') # Kinship (ids remapped in 03b)
# NB this matrix was sparsified in 03b: relatedness below 0.025 zeroed, 0.01 ridge on the
# diagonal. The raw matrix is 32% non-zero and makes every fit take hours. See the decision
# record in 03b for the sweep, the statistical implications, and what the manuscript needs
# to state. If fits are unexpectedly slow, check that this file is the sparsified one:
#   Matrix::nnzero(km) / nrow(km)^2   should be ~0.0013, not ~0.32
data <- data[NWD_Id %in% colnames(km)]
unsuitable_snps <- data[, names(.SD)[sapply(.SD, \(x) all(is.na(x)) | sum(x,na.rm=T)<10)], .SDcols=patterns('^chr')]
data <- data[, .SD, .SDcols=!unsuitable_snps]
cpaids <- setdiff(cpaids,unsuitable_snps)

data[ # dosages
  ][, .SD, .SDcols=variants_of_interest[cpaid %in% cpaids & grepl('KEW 2022',analysis),cpaid]
  ] |>
  tbl_summary(
    missing='no',
    statistic = list(
      all_continuous()  ~ c("{mean} ± {sd}"),
      all_categorical() ~ c("{n} ({p}%)"   )
  )) |> add_n() |> as_gt() |> gt:::as.tags.gt_tbl()

met_nms <- names(fread('data/derived/metabolomics/QCd/merged_QCd-aligned.csv',nrows=0,drop=1))
covars <- c('sex', 'age', 'smoking', 'metabolomics_visit', paste0('gPC',1:9)) # 9 gPCs, as in 02b.
stopifnot('missing gPCs -- check 03b merged freeze9_pcair_results' =
          all(paste0('gPC',1:9) %in% names(data)))

# FHS metabolomics here is cross-sectional -- one sample per person -- so the models carry
# only the kinship-structured intercept. 02b additionally has an iid permanent-individual
# term for MESA's repeated exams; that would be confounded with the residual here. FHS
# relatedness is denser than MESA's, so this single term does correspondingly more work.
stopifnot('FHS frame is not one row per person -- revisit the RE structure' =
          data[, .N, by=NWD_Id][, max(N)] == 1)
outcomes <- c('hdl_log', 'tg_log', 'fg')
exposures <- c('bmi','sex')

calc_eff_n_metabolites <- \(met_nms) {
  met_nms <- met_nms[!is.na(met_nms)] |> unique()
  met_mtx <- data[ rowSums(is.na(data[,..met_nms]))==0, ..met_nms ] |> as.matrix(rownames=1) # Only samples having data for ALL metabolomics methods
  met_eigvals <- prcomp(met_mtx, scale=T, center=T)$sdev^2
  met_eff_n <- sum(met_eigvals)^2 / sum(met_eigvals^2)
}

# Map aligned vs. MESA-specific metabolites
This code was used to map MESA -> FHS metabolites, so that we only rerun the metabolites that were already significant in MESA. But now, we want to run FHS from scratch regardless of what we found in MESA. Hence why this piece is now commented out.

Not all can be mapped, but we try our best.

In [ ]:
#met_info_mesa  <- fread('data/derived/metabolomics/met_info-mesa.csv'   )
#met_info_align <- fread('data/derived/metabolomics/met_info-aligned.csv')
#
## Fixing differences in the formats I made for these two files:
## * "_pos"/"_neg" at the end of the unique_met_ids
## * "Amide-neg" vs. "Amide-Negative-sMRM" Method name
#met_info_align[Method=='Amide-Negative-sMRM', Method := 'Amide-neg']
#met_info_mesa [, unique_met_id := sub('_pos','',sub('_neg','',unique_met_id))]
#
#signif_in_mesa <- fread('runs.csv')[grepl('chr14',snp) & !single_exam & (signif_funnel | signif_mwis), met]
#signif_in_mesa <- sub('_pos','',sub('_neg','',signif_in_mesa)) # "_pos"/"_neg" at the end of ids fix (messy)
#signif_in_mesa <- met_info_mesa[unique_met_id %in% signif_in_mesa]
#
#met_info_align$unique_met_id %in% signif_in_mesa$unique_met_id |> sum() # Aw that's only like half :(
#tmp1 <- met_info_align[Method!='Amide-neg'][signif_in_mesa[Method!='Amide-neg'], on=.(unique_met_id)]
#tmp2 <- met_info_align[Method=='Amide-neg'][signif_in_mesa[Method=='Amide-neg'], on=.(  Metabolite )][, unique_met_id := i.unique_met_id] # I manually generated the unique ids for the amines arbitrarily. Must match on name instead.
#tmp <- rbind(tmp1,tmp2,fill=T)
#
## Sanity check that the QI ids do correspond between datasets, its just that not all the MESA metabolites are in the aligned dataset.
#tmp[, all(      Compound_ID==      i.Compound_ID | is.na(      Compound_ID))]
#tmp[, all(Pilot_Compound_ID==i.Pilot_Compound_ID | is.na(Pilot_Compound_ID))]
#
#met_nms <- tmp[Method=='Amide-neg' | !is.na(Compound_ID), unique_met_id]
#
#met_nms <- met_nms[met_nms %in% names(data)] # To account for metabolites lost during QC for high missingness.

# Confirm known associations, choose SNPs to pursue
## Define formulas

In [ ]:
runs1 <- CJ(sorted=F,
  Y = outcomes,
  E = exposures,
  M = NA,
  G = transpose(variants_of_interest[cpaid %in% cpaids, .(cpaid,analysis,exposure,outcome)]),
  model=c('Y~E', 'E~G', 'Y~G', 'Y~G*E', 'Y~G*E_no_G*sex'),
  exam='all', lm_or_lmm='LMM'
)[

  ][, c('G','G_analysis','G_exposure','G_outcome') := transpose(G)

  # Rm per-snp/per-outcome rows for models w/o a snp/outcome term
  ][!grepl('Y', model), `:=`(Y=NA)
  ][!grepl('G', model), `:=`(G=NA, G_analysis=NA, G_exposure=NA, G_outcome=NA) 
  ][!grepl('E', model), `:=`(E=NA)
  ][!duplicated(paste(model,Y,E,G))

  # Filter all the possible combinations of models, down to the ones we want.
  ## For each row, if model has G term, only keep if the Y & E are relevant to that SNP.
  ][mapply(model,Y,E,G_outcome,G_exposure, FUN=\(model,Y,E,G_outcome,G_exposure) {
      fifelse(!grepl('G',model), T,
        fifelse(!grepl('Y',model), T, Y %in% strsplit(G_outcome ,' ')[[1]]) &
        fifelse(!grepl('E',model), T, E %in% strsplit(G_exposure,' ')[[1]])
      )
    })
  ][is.na(E) | !(model=='Y~G*E_no_G*sex' & E=='sex')
  ][is.na(E) | E != 'smoking' # Categorical variable adds complication. Not worth it for just 1 SNP.

  ][model=='Y~E',            fmla := paste0('<Y> ~ <E>')
  ][model=='E~G',            fmla := paste0('<E> ~ <G>')
  ][model=='Y~G',            fmla := paste0('<Y> ~ <G>')
  ][model=='Y~G*E',          fmla := paste0('<Y> ~ <G>*<E> + <G>*sex')
  ][model=='Y~G*E_no_G*sex', fmla := paste0('<Y> ~ <G>*<E>')

  # Finalize
  ][, fmla := paste(fmla, '+', paste(covars,collapse='+'))   # Add covars common to all models.
  ][model=='E~G', fmla := gsub('\\*<E>','',fmla)             # E~G shouldn't have E itself in the covariates.
  ][model=='Y~G', fmla := gsub('\\*<E>','',fmla)             # Y~G shouldn't have E        in the covariates.
  ][grepl('Y~G\\*E',model), fmla := paste(fmla, '+',         # Add gPC*E term for models w/ G*E focal term.
     paste(collapse='+', paste0('gPC',1:9,'*<E>')))
  ][, fmla := Map(Y,fmla, f=\(Y,fmla) gsub('<Y>',Y,fmla))    # Replace placeholders.
  ][, fmla := Map(G,fmla, f=\(G,fmla) gsub('<G>',G,fmla))    #
  ][, fmla := Map(E,fmla, f=\(E,fmla) gsub('<E>',E,fmla))    #
  ][, fmla := paste(fmla, '+', '(1|NWD_Id)')                 # Add random intercept.

  # Specify the term of interest whose stats will be extracted by specifying a pattern to grep.
  #   Interaction terms may be named either "X:Y" or "Y:X", need to account for both....
  ][model %in% c('Y~E'),    term_pat := paste0('^',E,'$')
  ][model %in% c('E~G'),    term_pat := paste0('^',G,'$')
  ][model %in% c('Y~G'),    term_pat := paste0('^',G,'$')
  ][grepl('Y~G\\*E',model), term_pat := paste0(G,':',E,'|',E,':',G)
  ][is.na(term_pat),        term_pat := 'NONE' # (grepping by NA would cause problems.)
]

## Confirm Y ~ E assocations
Model: `Y ~ E + covariates + (1|ID)`
Only top 10 model terms are shown. E is highlighted.

In [ ]:
tbls <- runs1[model=='Y~E', .(Map(fmla,Y,E, f=\(fmla,Y,E) {
  myLmer(fmla, data, list(NWD_Id=km)) |>
  summary() |> coef() |>
  as.data.table(keep.rownames='term') |>
  (\(dt) dt[order(`Pr(>|t|)`)])() |>
  head(10) |> gt(caption=paste(Y,'~',E)) |>
  gt_highlight_rows(rows=grepl(E,term)) |>
  gt:::as.tags.gt_tbl()
}))]

tbls$V1

## Confirm Y ~ G associations

In [ ]:
options(mc.cores=4L)
i <- 0
runs1[
  ][model=='Y~G' & exam=='all'
    , c('est','se','df','t','p','n') := transpose(mcMap(fmla,term_pat, f=\(fmla,pat) {
        message('  ',(i<<-i+1)*getOption('mc.cores'),'/',.N,'\r',appendLF=F) # Progress bar
        model <- myLmer(fmla,data,list(NWD_Id=km))
        coefs <- summary(model)$coefficients
        c(coefs[grepl(pat,rownames(coefs)),],
          nrow(model@frame)                 )
      }))
]

In [ ]:
# Display
variants_of_interest[
  ][runs1, on=.(cpaid=G)
  ][model=='Y~G' & exam=='all'
  ][order(p)
  ][, .(Y,G=cpaid,E,Gene=gene,rsID=rsid,analysis,Annotation=annotation,G_p=signif(p,3), G_β=signif(est,3),`N (all timepoints)`=n)
] |> gt(caption='G main effects with p<0.05 are highlighted') |> gt_highlight_rows(rows=G_p<0.05) |> gt:::as.tags.gt_tbl()

## Confirm Y ~ GxE associations
Model: `Y ~ G*E + G*sex + covars + (1|ID)`

In [ ]:
options(mc.cores=4L)
i <- 0
runs1[
  ][model=='Y~G*E' & exam=='all'
    , c('est','se','df','t','p','n') := transpose(mcMap(fmla,term_pat, f=\(fmla,pat) {
        message('  ',(i<<-i+1)*getOption('mc.cores'),'/',.N,'\r',appendLF=F) # Progress bar
        model <- myLmer(fmla,data,list(NWD_Id=km))
        coefs <- summary(model)$coefficients
        c(coefs[grepl(pat,rownames(coefs)),],
          nrow(model@frame)                 )
      }))
]

In [ ]:
# Display
variants_of_interest[
  ][runs1, on=.(cpaid=G)
  ][model=='Y~G*E' & exam=='all'
  ][order(p)
  ][, .(Y,G=cpaid,E,Gene=gene,rsID=rsid,analysis,Annotation=annotation,GxE_p=signif(p,3), GxE_β=signif(est,3),`N (all timepoints)`=n)
] |> gt(caption='GxEs with p<0.05 are highlighted') |> gt_highlight_rows(rows=GxE_p<0.05) |> gt:::as.tags.gt_tbl()

# Replicate the MESA GxM hits

02b writes `results/mesa_signif_GxMs.csv`: the metabolite x SNP interactions that passed
the metabolome-wide screen in MESA. This section tests those same interactions in FHS and
nothing else -- a handful of models rather than a second screen, which is what makes these
fits affordable given how dense the FHS kinship matrix is.

Two restrictions, both deliberate:

* **Primary GxM interaction only.** No `G:E` adjustment and no mediation analysis. The
  `G:E` term is a single commented line in the formula below if that changes.
* **Kinship-only random effect.** FHS metabolomics is cross-sectional, one sample per
  person, so there is no repeated-measures term to carry. 02b additionally fits an iid
  permanent-individual effect for MESA's repeated exams; here that would be confounded
  with the residual. Setup asserts the one-row-per-person assumption.

In [ ]:
grab_file_if_not_extant('results/mesa_signif_GxMs.csv')   # Written by 02b.
mesa_hits <- fread('results/mesa_signif_GxMs.csv')
setnames(mesa_hits, '\u03b2_GxM', 'mesa_beta', skip_absent=TRUE)  # ASCII name for the beta column.
setnames(mesa_hits, 'p', 'mesa_p', skip_absent=TRUE)

# MESA ids carry an ionization suffix the aligned dataset does not:
#   QI14545_C18_neg -> QI14545_C18
# Amide-neg would NOT be mappable this way -- 01a builds those ids from a row index, so they
# do not correspond across datasets at all -- but no Amide-neg feature reached the hit list,
# so every hit here is addressable. The check below catches it if that ever changes.
mesa_hits[, M_aligned := sub('_(pos|neg)$', '', M)]

# Two distinct reasons a hit may be untestable. They mean different things, so they are
# counted separately rather than as one 'missing' total.
mesa_hits[, met_present      := M_aligned %in% met_nms]
mesa_hits[, exposure_present := E %in% names(data)]
mesa_hits[, testable         := met_present & exposure_present]

cat('MESA hits:', nrow(mesa_hits), '| testable in FHS:', mesa_hits[, sum(testable)], '\n\n')
print(mesa_hits[, .(hits=.N, testable=sum(testable),
                    metabolite_absent=sum(!met_present),
                    exposure_absent=sum(!exposure_present)),
                by=.(rsid, gene, Y, E)])

if (mesa_hits[, any(!exposure_present)])
  # NB `mesa_hits[!exposure_present]` reads as data.table's not-join syntax, not as
  # logical negation, and fails looking the symbol up in the calling scope.
  cat('\nNOTE:', paste(mesa_hits[exposure_present == FALSE, unique(E)], collapse=', '),
      'is absent from the FHS frame, so those anchors cannot be replicated here.\n',
      ' Physical activity is searched for in 00_picsure but never coalesced into a named\n',
      ' column, so it never reaches analysis_df-fhs. Nothing below is hard-coded to a\n',
      ' particular exposure: add the variable upstream and those hits become testable.\n')

hits <- mesa_hits[testable == TRUE]
stopifnot('no MESA hit is testable in FHS -- check the id mapping and the exposures' =
          nrow(hits) > 0)

In [ ]:
# Match 02b's MWIS specification as closely as FHS allows: same covariates, the same G:sex
# term, the same gPC x M interactions -- so an estimate here is comparable to the MESA one
# rather than merely similar. The one intentional omission is the G:E adjustment; uncomment
# the marked line to restore it.
hits[, fmla := paste0(Y, ' ~ ', G, '*', M_aligned)
  ][, fmla := paste(fmla, '+', paste(covars, collapse='+'))
  ][, fmla := paste(fmla, '+', paste0(G, ':sex'))
  ][, fmla := paste(fmla, '+', paste(collapse='+', paste0('gPC', 1:9, '*', M_aligned)))
  # ][, fmla := paste(fmla, '+', paste0(G, '*', E))   # G:E adjustment -- descoped, see above.
  ][, fmla := paste(fmla, '+ (1|NWD_Id)')
  ][, term_pat := paste0(G, ':', M_aligned, '|', M_aligned, ':', G)
]

options(mc.cores = min(4L, nrow(hits)))
i <- 0
hits[, c('fhs_est','fhs_se','fhs_df','fhs_t','fhs_p','fhs_n') :=
  transpose(mcMap(fmla, term_pat, f = \(fmla, pat) {
    message('  ', (i <<- i+1)*getOption('mc.cores'), '/', .N, '\r', appendLF=FALSE)
    model <- myLmer(fmla, data, list(NWD_Id=km))
    coefs <- summary(model)$coefficients
    c(coefs[grepl(pat, rownames(coefs)), ], nrow(model@frame))
  }))]

## Replication summary

These are pre-specified hypotheses rather than a screen, so the yardstick is Bonferroni
over the hits actually tested, not the metabolome-wide threshold 02b used. Direction is
reported separately from significance: agreeing in sign without reaching the threshold is
a weaker but distinct statement, and worth being able to see.

In [ ]:
p_replication <- 0.05 / nrow(hits)

rep_tbl <- hits[
  ][, same_direction := sign(fhs_est) == sign(mesa_beta)
  ][, replicated     := same_direction & fhs_p < p_replication
  ][order(fhs_p)
  ][, .(rsid, gene, outcome=Y, exposure=E, metabolite=met_label,
        MESA_beta=signif(mesa_beta,3), MESA_p=signif(mesa_p,3),
        FHS_beta=signif(fhs_est,3), FHS_se=signif(fhs_se,3), FHS_p=signif(fhs_p,3),
        n=fhs_n, same_direction, replicated)
]

cat('Replication threshold: p <', signif(p_replication,3),
    ' (Bonferroni over', nrow(hits), 'tested hits)\n')
cat('Same direction as MESA:', rep_tbl[, sum(same_direction)], 'of', nrow(rep_tbl),
    '| replicated:', rep_tbl[, sum(replicated)], '\n\n')

rep_tbl |> gt() |> tab_header('FHS replication of the MESA GxM hits') |> gt:::as.tags.gt_tbl()

## Write

In [ ]:
fwrite(rep_tbl,   'results/fhs_replication.csv')       # Tested hits, MESA beside FHS.
fwrite(mesa_hits, 'results/fhs_replication_full.csv')  # All hits, including why any was skipped.

system(paste0('gcloud storage cp results/fhs_replication*.csv ', ws_bucket, '/results/'))